In [73]:
# --- STEP 1: SAFE INSTALL FOR MAC ---
%pip install pandas numpy scikit-learn folium

# --- STEP 2: IMPORTS ---
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
import folium
from folium.plugins import HeatMap

# --- STEP 3: DATA & ANTI-LEAKAGE ---
# Load your specific Delhi dataset
df = pd.read_csv("delhi_ncr_aqi_dataset.csv")

# PREVENTION: Shift PM2.5 to predict 12 hours into the future
df['target_pm25'] = df['pm25'].shift(-12)

# CLEANING: Drop 'aqi' (Target Leakage) and empty rows
df_clean = df.dropna(subset=['target_pm25'])
features = ['temperature', 'humidity', 'wind_speed', 'latitude', 'longitude']

X = df_clean[features]
y = df_clean['target_pm25']

# --- STEP 4: TRAIN THE PREDICTIVE ENGINE ---
# We use Random Forest as requested in the "Breath-Analyzer" mission
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

print("✅ Model Trained Successfully without Data Leakage!")

# --- STEP 5: VISUALIZATION (THE PROTOTYPE) ---
# Center the map on Delhi
m = folium.Map(location=[28.6139, 77.2090], zoom_start=11)

# Add the Heatmap (Red Zones & Green Corridors)
heat_data = [[row['latitude'], row['longitude'], row['pm25']] for index, row in df.iterrows()]
HeatMap(heat_data).add_to(m)

m.save("index.html")
print("✅ Heatmap generated as 'index.html'. Open this file to see your prototype!")

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
✅ Model Trained Successfully without Data Leakage!
✅ Heatmap generated as 'index.html'. Open this file to see your prototype!


In [74]:
%pip install "urllib3<2.0"

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [75]:
# ADD THIS to your Visualization Cell before m.save
folium.Marker(
    location=[28.6139, 77.2090],
    popup="<b>ALERT:</b> Predicted PM2.5 > 150. Avoid this junction!",
    icon=folium.Icon(color='red', icon='info-sign')
).add_to(m)

m.save("breath_analyzer_output.html")

In [76]:
import numpy as np
from scipy.spatial.distance import cdist

# --- NEW IDW LOGIC ---
def estimate_aqi_idw(target_lat, target_lon, sensor_df, power=2):
    """Estimates AQI for a 'Blind Spot' using Inverse Distance Weighting."""
    # Calculate distances from target to all sensors
    coords = sensor_df[['latitude', 'longitude']].values
    target = np.array([[target_lat, target_lon]])
    distances = cdist(coords, target).flatten()
    
    # Avoid division by zero if point is exactly on a sensor
    distances[distances == 0] = 0.001 
    
    # Calculate weights
    weights = 1 / (distances ** power)
    
    # Calculate weighted average
    estimate = np.sum(weights * sensor_df['pm25']) / np.sum(weights)
    return estimate

# --- UPDATED VISUALIZATION ---
m = folium.Map(location=[28.6139, 77.2090], zoom_start=11)

# 1. Add heatmap of known sensors
heat_data = [[row['latitude'], row['longitude'], row['pm25']] for index, row in df.iterrows()]
folium.plugins.HeatMap(heat_data).add_to(m)

# 2. Add 'Blind Spot' Estimation Markers
# Example: Let's estimate a known blind spot in Central Delhi
blind_spots = [
    {"name": "Blind Spot A", "lat": 28.6500, "lon": 77.2300},
    {"name": "Blind Spot B", "lat": 28.5800, "lon": 77.3000}
]

for spot in blind_spots:
    est_value = estimate_aqi_idw(spot['lat'], spot['lon'], df)
    color = 'red' if est_value > 150 else 'green'
    
    folium.Marker(
        location=[spot['lat'], spot['lon']],
        popup=f"<b>{spot['name']} (Estimated)</b><br>PM2.5: {est_value:.2f}",
        icon=folium.Icon(color=color, icon='info-sign')
    ).add_to(m)

m.save("index.html")
print("✅ AeroGuard Intelligence Map generated with IDW estimation!")

✅ AeroGuard Intelligence Map generated with IDW estimation!


In [77]:
import numpy as np
import folium
from folium.plugins import HeatMap

# 1. Define the Boundries of Delhi-NCR (The "Outer Region")
lat_min, lat_max = 28.3, 28.9
lon_min, lon_max = 76.8, 77.5

# 2. Create a Dense Grid of Coordinates
grid_size = 60  # Increase to 100 for higher resolution, but it will be slower
lat_grid = np.linspace(lat_min, lat_max, grid_size)
lon_grid = np.linspace(lon_min, lon_max, grid_size)

import branca.colormap as cm

# 1. Define the Standard AQI Color Scale for the Legend
# 0-50: Good, 51-100: Satisfactory, 101-200: Moderate, 201-300: Poor, 301+: Severe
colormap = cm.StepColormap(
    colors=['#00e400', '#ffff00', '#ff7e00', '#ff0000', '#8f3f97', '#7e0023'],
    index=[0, 50, 100, 200, 300, 500], # AQI Buckets
    vmin=0, vmax=500,
    caption='AQI (PM2.5) Risk Levels'
)

# 3. Standard AQI Color Coding Logic (Standard Notation)
# 0-30: Good (Green), 31-60: Satisfactory (Yellow), 61-90: Moderate (Orange)
# 91-120: Poor (Red), 121+: Severe (Purple/Maroon)
# 2. Convert that scale into a gradient for the HeatMap
# We add '0.0' as transparent so the map doesn't have a solid background
    
gradient_map = {
    0.0: "rgba(0,228,0,0)",   # Transparent for very low values
    0.1: "#00e400",          # Green (Good)
    0.2: "#ffff00",          # Yellow (Satisfactory)
    0.4: "#ff7e00",          # Orange (Moderate)
    0.6: "#ff0000",          # Red (Poor)
    0.8: "#8f3f97",          # Purple (Very Poor)
    1.0: "#7e0023"           # Maroon (Severe)
}

# 3. Generate the Map with the Legend
m_coverage = folium.Map(location=[28.6139, 77.2090], zoom_start=11, tiles='cartodbpositron')

# 4. Generate Interpolated Data for the "Whole Map"
interpolated_data = []

print("🔄 Generating AeroGuard Coverage Surface... Please wait.")

for lat in lat_grid:
    for lon in lon_grid:
        # Use your existing IDW function to estimate AQI at this grid point
        estimate = estimate_aqi_idw(lat, lon, df)
        # We append [lat, lon, weight] for the HeatMap
        interpolated_data.append([lat, lon, estimate])

# 5. Create the Map
# --- STEP 3 & 5: Create the Map ONCE ---
m_coverage = folium.Map(location=[28.6139, 77.2090], zoom_start=11, tiles='cartodbpositron')

# --- STEP 4: Generate Interpolated Data ---
interpolated_data = []  # Force reset to avoid accumulation
grid_size = 100         # 50x50 is perfect for a 5MB file
lat_grid = np.linspace(28.3, 28.9, grid_size)
lon_grid = np.linspace(76.8, 77.5, grid_size)

for lat in lat_grid:
    for lon in lon_grid:
        estimate = estimate_aqi_idw(lat, lon, df)
        interpolated_data.append([lat, lon, estimate])

# --- STEP 6: Add the Surface ---
folium.plugins.HeatMap(
    interpolated_data,
    gradient=gradient_map,
    min_opacity=0.15,
    max_val=400,    
    radius=22, blur=15
).add_to(m_coverage)

# --- NEW STEP: ATTACH THE SCALE (This was missing!) ---
# This adds the professional color bar to the map
colormap.add_to(m_coverage)

# --- STEP 7 FIX: Plot ONLY Unique Sensor Locations ---
# REMOVE the old df.iterrows() loop. Use this instead:
unique_sensors = df.drop_duplicates(subset=['latitude', 'longitude'])

for _, row in unique_sensors.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3, color='black', fill=True, weight=1
    ).add_to(m_coverage)

# Save the map
m_coverage.save("index.html")
print("✅ SUCCESS: File size should now be ~5MB and scale is visible!")

🔄 Generating AeroGuard Coverage Surface... Please wait.
✅ SUCCESS: File size should now be ~5MB and scale is visible!


/var/folders/jd/szf9hjc966b28rs2gwv4zxxc0000gn/T/ipykernel_34362/4247573736.py:72: UserWarning: The `max_val` parameter is no longer necessary. The largest intensity is calculated automatically.
  folium.plugins.HeatMap(


In [78]:
# 'm' is your map object created with folium
m.save("breath_analyzer_output.html") 
print("HTML file generated successfully!")

HTML file generated successfully!
